<a href="https://colab.research.google.com/github/Aimagicians/ai-agents-tutorial/blob/main/Copy_of_agents_chat_to_agent_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 Building AI Agents: From Chat to Tool Use to Agent Loop

IMPORTANT: Make a copy of this notebook:
1. Menu: File > Save a copy in Drive
2. It will open a new tab, in that new tab click on "🗝️" or key icon in sidebar. Enter "OPENAI_API_KEY" and paste the API Key in password field
3. Then click "Run" or play-icon for each cell one-by-one. Make sure you do NOT miss any cell

---

Get your API key for Tavily at: https://www.tavily.com/

---

## What you'll build in this notebook:

| Section | What | Key Insight |
|---|---|---|
| 1 | Chat interface | LLM is stateless — *you* manage history |
| 2 | Chat + Search tool | LLM *requests* tool calls — *your code* executes them |
| 3 | The Agent Loop | A `while` loop is the entire difference between chat and agent |
| 4 | Agent + Skill | Same loop, same tools — instructions change everything |

> **Core thesis:** An agent is just an LLM in a loop with tools. Everything else is orchestration.

**Stack:** OpenRouter/OpenAI (API gateway) → OpenAI SDK (client) → Gemini 2.5 Flash (model) → Tavily (search tool)


---
## 0. Setup

### API Keys Required
1. **OpenRouter** — [Get key at openrouter.ai/keys](https://openrouter.ai/keys) (add some credits or use free models)
2. **Tavily** — [Sign up for free tier](https://app.tavily.com) (1000 searches/month)


In [ ]:
# Install dependencies
!pip install -q openai tavily-python

In [ ]:
# API Keys — paste yours here
import os
from google.colab import userdata


OPENROUTER_API_KEY = "PUT_YOUR_API_KEY_HERE"
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

# os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
# os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

assert OPENROUTER_API_KEY, "⚠️ Paste your OpenRouter API key above"
assert TAVILY_API_KEY, "⚠️ Paste your Tavily API key above"
print("✅ Keys set")

✅ Keys set


In [ ]:
# Initialize clients
from openai import OpenAI
from tavily import TavilyClient

# OpenRouter is OpenAI-compatible — just point base_url at OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

tavily = TavilyClient(api_key=TAVILY_API_KEY)

# Pick a model — any OpenRouter model slug works here
MODEL = "gpt-5.4-mini"  # cheap, fast, good at tool calling

# Quick test
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Which model are you?"}]
)
print(test.choices[0].message.content)

I’m ChatGPT, an OpenAI language model.


---
## 1. The Chat Interface

The simplest thing you can build with an LLM: a loop that takes input, sends it to the model, prints the response.

**But there's a catch.** LLMs are stateless — they don't remember previous messages. *You* have to send the full conversation history every time.


In [ ]:
# 1a. Single turn — no memory
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Sid."}]
)
print(response.choices[0].message.content)

Nice to meet you, Sid. How can I help?


In [ ]:
# Now ask it what your name is — it won't know
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What's my name?"}]
)
print(response.choices[0].message.content)
# 👆 It doesn't remember. Each call is independent.

I don’t know your name unless you tell me. If you want, share it and I’ll use it.


### The fix: maintain a conversation history

We store messages in a list and send the *entire* list with every API call. The model sees the full context each time.


In [ ]:
# 1b. Chat with history — context persists across cells

# This list IS the memory. It persists across cells.
chat_history = []

def chat(user_message: str) -> str:
    """Send a message, get a response, maintain history."""
    # Add user message to history
    chat_history.append({"role": "user", "content": user_message})

    # Send full history to the model
    response = client.chat.completions.create(
        model=MODEL,
        messages=chat_history
    )

    assistant_msg = response.choices[0].message.content

    # Add assistant response to history
    chat_history.append({"role": "assistant", "content": assistant_msg})

    return assistant_msg

In [ ]:
# Now it remembers
print(chat("My name is Sid."))

Nice to meet you, Sid. How can I help?


In [ ]:
print(chat("What's my name?"))
# 👆 It knows — because we sent the full history

Your name is Sid.


In [ ]:
# Check what's actually being sent
print(f"Messages in history: {len(chat_history)}")
for msg in chat_history:
    print(f"  [{msg['role']}] {msg['content'][:80]}")

Messages in history: 4
  [user] My name is Sid.
  [assistant] Nice to meet you, Sid. How can I help?
  [user] What's my name?
  [assistant] Your name is Sid.


### 🔑 Takeaway

The LLM is stateless. **You** are the memory. The `chat_history` list *is* the conversation — the model just responds to whatever you send it.

This is exactly how ChatGPT, Claude, Gemini chat apps work under the hood. There's a list of messages. That's it.


---
## 2. Adding a Tool: Web Search

The chat works, but the LLM can only use what's in its training data. Ask it about today's stock price and it'll hallucinate or refuse.

**Tools** let the LLM request external actions. The key insight:

> The LLM **never executes** anything. It *requests* a tool call. **Your code** executes it.

The flow:
1. User asks a question
2. LLM decides it needs a tool → returns a **tool call** (not text)
3. Your code runs the function with the LLM's arguments
4. You send the result back to the LLM
5. LLM uses the result to generate a text response


### 2a. Define the tool

We need two things:
1. **The actual function** — runs Tavily search
2. **The schema** — tells the LLM what the tool does (JSON spec)


In [ ]:
import json

# 1. The actual function that executes search
def execute_search(query: str, max_results: int = 3) -> str:
    """Run a Tavily search and return formatted results."""
    response = tavily.search(query=query, max_results=max_results)
    results = []
    for r in response["results"]:
        results.append(f"**{r['title']}**\n{r['content']}\nSource: {r['url']}\n")
    return "\n---\n".join(results)

# Quick test — does Tavily work?
print(execute_search("NVIDIA stock price today", max_results=2))

**NVIDIA (NVDA) Stock Price Today, News, Chart, Earnings | Coinbase**
NVIDIA (NVDA) stock price today is $214.80 - $221.01. See NVIDIA news, earnings, and stock charts on Coinbase.
Source: https://www.coinbase.com/stock/NVDA

---
**NVDA Stock Price | Analyst Target 301.32 & Consensus - eToro**
A 'stable' price indicates that the change in weekly price is between -0.5% and 0.5%. The NVIDIA Corporation share price today is ‎$‎215.33, reflecting a
Source: https://www.etoro.com/markets/nvda



In [ ]:
# 2. The tool SCHEMA — this is what the LLM reads
# It never sees your Python code. It only sees this JSON description.

tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Use this for any question about recent events, live data, stock prices, news, or anything requiring up-to-date information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query, make sure you include your keywords here"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Number of results to return (1-5). Default 3."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("✅ Tool schema defined")
print(json.dumps(tools[0]["function"], indent=2))

✅ Tool schema defined
{
  "name": "web_search",
  "description": "Search the web for current information. Use this for any question about recent events, live data, stock prices, news, or anything requiring up-to-date information.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "The search query, make sure you include your keywords here"
      },
      "max_results": {
        "type": "integer",
        "description": "Number of results to return (1-5). Default 3."
      }
    },
    "required": [
      "query"
    ]
  }
}


### 2b. Chat with tool use — single turn

Now we send the tool schema alongside the conversation. The LLM can *choose* to call the tool or respond directly.

**Watch the raw response** — when the LLM wants to use a tool, it returns `tool_calls` instead of `content`.


In [ ]:
# Fresh history for the tool-use chat
tool_chat_history = []

# Map of tool names to functions — this is our "runtime"
TOOL_REGISTRY = {
    "web_search": execute_search,
}

def chat_with_tools(user_message: str) -> str:
    """Chat that supports one round of tool use."""

    # Add user message
    tool_chat_history.append({"role": "user", "content": user_message})

    # Call the model WITH the tool schema
    response = client.chat.completions.create(
        model=MODEL,
        messages=tool_chat_history,
        tools=tools,
    )

    msg = response.choices[0].message

    # Save the model's response to history
    tool_chat_history.append(msg.model_dump())

    # Check: did the model return tool calls or text?
    if msg.tool_calls:
        print(msg.tool_calls)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"🔧 TOOL CALL: {fn_name}({fn_args})")

            # YOUR CODE executes the function — not the LLM
            fn = TOOL_REGISTRY.get(fn_name)
            if fn:
                result = fn(**fn_args)
            else:
                result = f"Unknown tool: {fn_name}"

            print(f"📋 Got {len(result)} chars of results\n")

            # Send the result back as a tool response
            tool_chat_history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

        # Get the model's final response using the tool results
        followup = client.chat.completions.create(
            model=MODEL,
            messages=tool_chat_history,
            tools=tools,
        )
        followup_msg = followup.choices[0].message
        tool_chat_history.append(followup_msg.model_dump())
        return followup_msg.content

    # No tool call — just return the text
    return msg.content

In [ ]:
# Ask something that DOESN'T need search
print(chat_with_tools("What is the capital of France?"))
print()
print("---")
print(f"History length: {len(tool_chat_history)} messages")

The capital of France is **Paris**.

---
History length: 2 messages


In [ ]:
# Now ask something that DOES need search
print(chat_with_tools("What's happening with NVIDIA stock today?"))

[ChatCompletionMessageFunctionToolCall(id='call_svZBWTHIWVGYfEas2VZz9gUv', function=Function(arguments='{"query":"NVIDIA stock today latest news price movement today", "max_results": 5}', name='web_search'), type='function', index=0)]
🔧 TOOL CALL: web_search({'query': 'NVIDIA stock today latest news price movement today', 'max_results': 5})
📋 Got 1349 chars of results

NVIDIA (NVDA) is down today, roughly **1.5%–1.8%**, trading around **$215–$220** based on current market snapshots.

What I found:
- Reuters shows **$219.51**, **-1.77%**
- TradingView shows about **$215.33**, **-1.64%**
- Robinhood shows about **$214.28**

A few likely reasons for the move:
- **Profit-taking / normal volatility** after a big run
- **Broader tech-sector weakness** can drag NVDA lower even without company-specific news
- **Volume is elevated**, which suggests active repositioning by traders

If you want, I can also dig into **the specific news catalyst behind today’s move** and summarize it in plain Engli

In [ ]:
# It remembers context too
print(chat_with_tools("Yeah please do that"))

🔧 TOOL CALL: web_search({'query': 'NVIDIA stock today news catalyst latest headlines today NVDA why down why up', 'max_results': 5})
📋 Got 1293 chars of results

Here are the main themes moving NVIDIA today:

- **Muted reaction despite strong fundamentals:** one market summary says NVDA is being driven by a mix of “blockbuster fundamentals” and a **muted investor reaction**, which has kept the stock under pressure.
- **Broader headline flow is mixed:** some coverage is pointing to **longer-term catalysts** like continued AI demand and possible **Chinese H200 chip orders** being prepared, but that’s more of a future support story than an immediate boost.
- **Price action:** latest quotes I found show NVDA around **$215**, down about **1.9%** on the day in one source.

In plain English: **today’s move looks more like profit-taking / a lukewarm market reaction than a fresh negative company-specific shock.**

If you want, I can also give you:
1. a **plain-English explanation of the latest 

### 🔧 Inspect the raw tool call

Let's look at exactly what the LLM returns when it wants to call a tool.


In [ ]:
# Make a fresh call and inspect the raw response
raw_response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is Tesla's stock price right now?"}],
    tools=tools,
)

msg = raw_response.choices[0].message

print(f"finish_reason: {raw_response.choices[0].finish_reason}")
print(f"content: {msg.content}")
print(f"tool_calls: {msg.tool_calls}")

if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"\n--- Tool Call ---")
        print(f"  id:        {tc.id}")
        print(f"  function:  {tc.function.name}")
        print(f"  arguments: {tc.function.arguments}")

# 👆 Notice: content is None. tool_calls has a structured request.
# The LLM is saying "I want to call web_search with these args"
# It CANNOT call the function. Only YOU can.

finish_reason: tool_calls
content: None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_yzZMbP3C8lsnVXxa9w2qO8K0', function=Function(arguments='{"query":"Tesla stock price TSLA right now", "max_results": 3}', name='web_search'), type='function', index=0)]

--- Tool Call ---
  id:        call_yzZMbP3C8lsnVXxa9w2qO8K0
  function:  web_search
  arguments: {"query":"Tesla stock price TSLA right now", "max_results": 3}


### 🔑 Takeaway

The tool use protocol is simple:
1. You tell the LLM what tools exist (schema)
2. The LLM *decides* to call one (returns `tool_calls`, not `content`)
3. **Your code** executes it
4. You send results back as `role: "tool"` messages
5. The LLM synthesizes an answer

**The LLM never touches your systems.** It's a structured request, nothing more.


---
## 3. The Agent Loop

Our chat-with-tools has a limitation: **it handles only one round of tool calls per turn.**

Ask it to "research the impact of tariffs on Indian IT companies" and it'll do one search and stop. A real researcher would search, read, search again with better queries, compare sources...

**An agent is a chat that loops until it's done.**

```
while not done:
    response = llm(history)
    if response has tool calls:
        execute them
        add results to history
    else:
        done — the LLM decided it has enough info
```

That's it. That's the entire difference between a chatbot and an agent.


In [ ]:
def agent_loop(
    user_message: str,
    system_prompt: str = "You are a helpful assistant with access to web search.",
    max_iterations: int = 10,
    verbose: bool = True
) -> str:
    """
    The agent loop: LLM + tools + loop until done.

    The model can call tools multiple times. It stops when it
    returns text instead of a tool call.
    """

    # Fresh history for each agent run
    history = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        if verbose:
            print(f"\n{'='*60}")
            print(f"⚡ Iteration {iteration}")
            print(f"{'='*60}")

        # Call the model
        response = client.chat.completions.create(
            model=MODEL,
            messages=history,
            tools=tools,
        )

        msg = response.choices[0].message

        # Save model response to history
        history.append(msg.model_dump())

        # Check for tool calls
        if not msg.tool_calls:
            # No tool calls — the model is done
            if verbose:
                print(f"\n✅ Agent finished after {iteration} iterations")
                print(f"📊 Total messages in history: {len(history)}")
            return msg.content

        # Execute each tool call
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)

            if verbose:
                print(f"🔧 Tool call: {fn_name}({fn_args})")

            fn = TOOL_REGISTRY.get(fn_name)
            if fn:
                result = fn(**fn_args)
            else:
                result = f"Unknown tool: {fn_name}"

            if verbose:
                print(f"   → Got {len(result)} chars")

            # Send tool result back
            history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return "⚠️ Agent hit max iterations without completing."


### Test the agent

Let's give it a question that requires **multiple searches** to answer well.


In [ ]:
# Simple question — should take 1-2 iterations
result = agent_loop("What is NVIDIA's current stock price?")
print("\n" + "─"*60)
print("FINAL ANSWER:")
print("─"*60)
print(result)


⚡ Iteration 1
🔧 Tool call: web_search({'query': 'NVIDIA current stock price NVDA current price', 'max_results': 3})
   → Got 853 chars

⚡ Iteration 2

✅ Agent finished after 2 iterations
📊 Total messages in history: 5

────────────────────────────────────────────────────────────
FINAL ANSWER:
────────────────────────────────────────────────────────────
NVIDIA (NVDA) is currently trading around **$215.33** per share.

Note: stock prices change constantly, so if you want, I can also check the latest **after-hours** price or today’s **high/low**.


In [ ]:
# Complex question — should take multiple iterations
result = agent_loop(
    "Research the impact of US tariffs on Indian IT companies in 2025. "
    "Cover TCS, Infosys, and Wipro specifically."
)
print("\n" + "─"*60)
print("FINAL ANSWER:")
print("─"*60)
print(result)


⚡ Iteration 1
🔧 Tool call: web_search({'query': '2025 US tariffs impact on Indian IT companies TCS Infosys Wipro tariffs 2025 news analysis', 'max_results': 5})
   → Got 4750 chars

⚡ Iteration 2
🔧 Tool call: web_search({'query': '2025 TCS tariffs impact US demand revenue outlook Reuters April 2025', 'max_results': 5})
   → Got 3774 chars
🔧 Tool call: web_search({'query': '2025 Infosys tariffs impact US demand outlook Reuters April 2025', 'max_results': 5})
   → Got 4625 chars
🔧 Tool call: web_search({'query': '2025 Wipro tariffs impact US demand outlook Reuters April 2025', 'max_results': 5})
   → Got 2621 chars

⚡ Iteration 3

✅ Agent finished after 3 iterations
📊 Total messages in history: 9

────────────────────────────────────────────────────────────
FINAL ANSWER:
────────────────────────────────────────────────────────────
Here’s a concise research summary of the impact of US tariffs on Indian IT companies in 2025, focused on TCS, Infosys, and Wipro.

## Bottom line
US tariffs d

### 🔑 Takeaway

The agent loop is a `while` loop. The LLM decides:
- **What** to search for (query construction)
- **Whether** to search again (loop continuation)
- **When** it's done (returns text instead of a tool call)

You provided the tools and the loop. The LLM provided the judgment.

**That's an agent.** Everything else — frameworks, orchestration, memory — is ergonomics on top of this loop.


---
## 4. Skills: Shaping Agent Behavior

The agent works but it's generic. Ask it to research a stock and it'll do a decent job — but it won't follow a consistent framework, might miss important angles, and the output format will vary each time.

**A skill is a set of instructions that shapes *how* the agent uses its tools.**

Same agent loop. Same tools. Different system prompt → completely different behavior.


### 4a. Without a skill — generic research

In [ ]:
# Ask for stock research WITHOUT a skill
result_without_skill = agent_loop(
    "Should I invest in NVIDIA stock? No investment advice, just report",
    system_prompt="You are a helpful assistant with access to web search.",
    verbose=True
)
print("\n" + "─"*60)
print("ANSWER (without skill):")
print("─"*60)
print(result_without_skill)


⚡ Iteration 1
🔧 Tool call: web_search({'query': 'NVIDIA latest earnings guidance stock valuation recent news 2026', 'max_results': 5})
   → Got 1448 chars

⚡ Iteration 2

✅ Agent finished after 2 iterations
📊 Total messages in history: 5

────────────────────────────────────────────────────────────
ANSWER (without skill):
────────────────────────────────────────────────────────────
I can’t tell you whether you should invest, but I can report the current picture on NVIDIA from recent public sources:

- **Recent earnings have been strong.** A CNBC report on NVIDIA’s Q2 2026 noted the company beat expectations and said sales growth would remain above 50%.
- **Revenue has been very large and still growing.** Kiplinger reported Q1 2026 earnings of **$1.87/share** on **$81.62 billion revenue**, above analyst forecasts.
- **The stock has already run up a lot.** S&P Global noted NVIDIA’s stock was up about **174% since January 2024** in one recent preview, which suggests a lot of optimism may

### 4b. The Skill

Here's a skill file — a structured set of instructions that tells the agent *how* to approach stock research. This is what you'd see in a production system.


In [ ]:
STOCK_RESEARCH_SKILL = """
# Stock Research Analyst

## Role
You are a stock research analyst producing structured research reports.

## Process
Follow this sequence — do NOT skip steps:
1. Search for the stock's recent price action and news (last 30 days)
2. Search for the latest quarterly earnings / financial results
3. Search for analyst ratings, target prices, and consensus
4. Search for sector-level or macro risks relevant to this stock

Each step should be a SEPARATE search with a targeted query.

## Output Format
Structure your response exactly like this:

### [Company Name] ([Ticker]) — Research Report

**Current Price:** [price and date]

**📰 Recent Catalysts**
- Bullish: [list]
- Bearish: [list]

**📊 Financials Snapshot**
- Revenue, profit, key ratios from latest quarter

**🎯 Analyst Consensus**
- Average target price, buy/hold/sell ratings

**⚠️ Risk Factors**
- Company-specific and macro risks

**📋 Verdict**
[Your analysis with confidence level: High/Medium/Low]
[State what would change your view]

## Rules
- Search at least 4 times (one per step above)
- Never give buy/sell recommendations — frame as analysis
- Always cite sources
- Flag if any data is older than 30 days
"""

print(STOCK_RESEARCH_SKILL)


# Stock Research Analyst

## Role
You are a stock research analyst producing structured research reports.

## Process
Follow this sequence — do NOT skip steps:
1. Search for the stock's recent price action and news (last 30 days)
2. Search for the latest quarterly earnings / financial results
3. Search for analyst ratings, target prices, and consensus
4. Search for sector-level or macro risks relevant to this stock

Each step should be a SEPARATE search with a targeted query.

## Output Format
Structure your response exactly like this:

### [Company Name] ([Ticker]) — Research Report

**Current Price:** [price and date]

**📰 Recent Catalysts**
- Bullish: [list]
- Bearish: [list]

**📊 Financials Snapshot**
- Revenue, profit, key ratios from latest quarter

**🎯 Analyst Consensus**
- Average target price, buy/hold/sell ratings

**⚠️ Risk Factors**
- Company-specific and macro risks

**📋 Verdict**
[Your analysis with confidence level: High/Medium/Low]
[State what would change your view]



### 4c. With the skill — structured research

In [ ]:
# Same question, WITH the skill as system prompt
result_with_skill = agent_loop(
    "Should I invest in NVIDIA stock? Prepare research report. Use the skills available. I understand you can't provide stock advice, add fair warinings",
    system_prompt=STOCK_RESEARCH_SKILL,
    verbose=True
)
print("\n" + "─"*60)
print("ANSWER (with skill):")
print("─"*60)
print(result_with_skill)


⚡ Iteration 1
🔧 Tool call: web_search({'query': 'NVIDIA stock recent price action news last 30 days NVDA', 'max_results': 5})
   → Got 1355 chars
🔧 Tool call: web_search({'query': 'NVIDIA latest quarterly earnings financial results revenue profit margins Q1 2025 NVDA', 'max_results': 5})
   → Got 1622 chars
🔧 Tool call: web_search({'query': 'NVIDIA analyst ratings target price consensus NVDA latest', 'max_results': 5})
   → Got 1480 chars
🔧 Tool call: web_search({'query': 'NVIDIA sector macro risks AI chips export controls competition supply chain 2025', 'max_results': 5})
   → Got 5521 chars

⚡ Iteration 2
🔧 Tool call: web_search({'query': 'NVIDIA latest quarterly earnings May 2025 revenue net income EPS margins official results', 'max_results': 5})
   → Got 1575 chars

⚡ Iteration 3

✅ Agent finished after 3 iterations
📊 Total messages in history: 10

────────────────────────────────────────────────────────────
ANSWER (with skill):
───────────────────────────────────────────────────

### 4d. Compare the outputs

Run this cell to see both outputs side by side.


In [ ]:
from IPython.display import display, Markdown, HTML

display(HTML("<h3>❌ Without Skill</h3>"))
display(Markdown(result_without_skill))
display(HTML("<hr><h3>✅ With Skill</h3>"))
display(Markdown(result_with_skill))

I can’t tell you whether you should invest, but I can report the current picture on NVIDIA from recent public sources:

- **Recent earnings have been strong.** A CNBC report on NVIDIA’s Q2 2026 noted the company beat expectations and said sales growth would remain above 50%.
- **Revenue has been very large and still growing.** Kiplinger reported Q1 2026 earnings of **$1.87/share** on **$81.62 billion revenue**, above analyst forecasts.
- **The stock has already run up a lot.** S&P Global noted NVIDIA’s stock was up about **174% since January 2024** in one recent preview, which suggests a lot of optimism may already be priced in.
- **Market sensitivity is high.** With a valuation that has moved sharply and expectations elevated, the stock may react strongly to any slowdown in AI demand, margins, export restrictions, or guidance.

**Neutral summary:** NVIDIA appears to have strong fundamentals and momentum, but it is also a widely followed, expectation-heavy stock, so the risk side is mainly around valuation and future growth staying very high.

If you want, I can also give you a **balanced bull/bear fact sheet** on NVIDIA without recommending anything.

### NVIDIA Corporation (NVDA) — Research Report

**Current Price:** $222.64 as of May 23, 2026, 9:55 AM EDT. Source: CNBC quote page.  
*Note: This price is from a search result and may not reflect a real-time market feed.*

**📰 Recent Catalysts**
- **Bullish:**
  - NVIDIA reported **record Q1 FY2026 revenue of $44.1 billion**, up **69% YoY**, with strong data center demand. Source: NVIDIA earnings release.
  - The stock has traded near the upper end of its 52-week range (**$129.16 to $236.54**), suggesting sustained investor interest. Source: CNBC / Robinhood quote pages.
- **Bearish:**
  - Export-control risk remains a major issue, especially around sales to China and other restricted markets. Source: The Register; Sourceability.
  - Supply-chain constraints and advanced packaging capacity at partners like TSMC can delay product ramps. Source: industry commentary in search results.
  - *Warning:* several search results are dated **older than 30 days** or reference future-dated pages in the search index, so those should be treated cautiously.

**📊 Financials Snapshot**
- Latest reported quarter found in search: **Q1 FY2026**
- Revenue: **$44.1 billion**
- YoY revenue growth: **69%**
- Segment highlight: Data Center revenue was the key driver of growth.
- Gross margin guidance mentioned in the release: **74.8% GAAP / 75.5% non-GAAP** for the next quarter guidance period. Source: NVIDIA investor relations press release.
- *Note:* I could not reliably confirm net income and EPS from the available search snippets without risking inaccuracy, so I’m not filling those in.

**🎯 Analyst Consensus**
- Average target price found across sources: about **$294.22**
- Consensus rating: **Buy / Strong Buy**
- Analyst mix in one source: **57% Strong Buy, 41% Buy, 3% Hold, 0% Sell**. Source: Public.com / Marketscreener / Seeking Alpha search results.
- Implied upside from the quoted target versus current search price is roughly **32%**.
- *Warning:* target prices vary by source and can change quickly.

**⚠️ Risk Factors**
- **Company-specific:**
  - Heavy dependence on AI infrastructure spending continuing at a very high rate.
  - Customer concentration risk among large cloud and data center buyers.
  - China/export restrictions could limit addressable market and product configurations.
  - Valuation risk: the stock can remain expensive even when fundamentals are strong.
- **Macro / sector:**
  - Semiconductor cyclicality and investor sensitivity to capex cycles.
  - Geopolitical tensions and trade restrictions affecting chip supply chains.
  - Competition from AMD, custom ASICs, and in-house AI chips from large hyperscalers.
  - Interest-rate and sentiment risk: high-growth stocks can rerate sharply if the market de-risks.

**📋 Verdict**
NVIDIA still looks like a high-quality business with exceptional growth, dominant positioning in AI accelerators, and strong analyst support. The main question is not whether the company is executing well, but whether the current valuation already discounts a lot of that success.

**Confidence level: Medium**
What would change my view:
- A clear slowdown in data center growth or weaker guidance
- Evidence that export controls materially hurt revenue growth
- Margin compression from pricing pressure, competition, or supply constraints
- A major shift in AI infrastructure spending from capex expansion to digestion

**Fair warning:** This is research, not investment advice. NVIDIA is a volatile, high-expectation stock, so even strong earnings can lead to sharp price swings if results are merely “good” rather than “extraordinary.” If you want, I can also produce a **bull/base/bear scenario table** for NVDA.

### 🔑 Takeaway

A **skill** = structured instructions that shape how an agent uses its tools.

- Same LLM. Same tools. Same loop. **Different system prompt → different behavior.**
- The skill controlled: search strategy (4 targeted searches vs 1 generic), output structure, what to include/exclude, and quality standards.
- In production systems (Claude, ChatGPT, custom agents), skills are how you go from "generic assistant" to "domain expert."

This is the conceptual equivalent of:
- CrewAI's `Agent(role=..., goal=..., backstory=...)`
- LangChain's agent prompt templates
- Claude's skill files


---
## Recap: What You Built

```
  1. Chat           →  LLM + history list
  2. Chat + Tools   →  LLM + history + tool schema + execute-and-return
  3. Agent          →  LLM + history + tools + WHILE LOOP
  4. Skilled Agent  →  Agent + structured system prompt (skill)
```

**The agent loop is the core primitive.** Everything else — frameworks (CrewAI, LangGraph), patterns (multi-agent, RAG), infrastructure (memory, eval) — builds on this loop.

**Did you know?**
You just built an **agnet harness**. It is responsible for looping the agent, detecting-extracting-and-running tool calls > sends back results to the model and generates a response > does it till the loop breaks.

This is what Cursor, Claude Code, and other agent harness's do!

### What's Next

In the next section, we'll take this exact pattern and implement it using **CrewAI** — a framework that manages the loop, tool execution, and agent coordination for you. You'll see how the hand-rolled code maps to CrewAI's abstractions:

| You built | CrewAI equivalent |
|---|---|
| `system_prompt` / skill | `Agent(role, goal, backstory)` |
| `execute_search()` | `@tool` decorator |
| The `while` loop | CrewAI's execution engine |
| "Research this" | `Task(description, expected_output)` |

### Bonus: Swap any model

Because we're using OpenRouter, you can swap the model slug and everything still works:

```python
MODEL = "anthropic/claude-sonnet-4"       # or
MODEL = "deepseek/deepseek-chat-v3-0324"   # or
MODEL = "meta-llama/llama-4-maverick"      # or
MODEL = "google/gemini-2.5-flash"          # ← what we used
```

Same code, same tools, different brain. That's the power of the OpenAI-compatible API.
